# 05 - Generation

Ollama RAG: retrieve context, build a grounded prompt, generate an answer with
`qwen3:8b` (local, fast iteration — swapped for vLLM in production per CLAUDE.md).

Retrieval mode is swappable (`dense` / `hybrid` / `hybrid_rerank`), defaulting to
`hybrid_rerank` with **SEA-LION** (`aisingapore/SEA-LION-E5-Embedding-600M`) as
the reranker — the final choice per `04b_retrieval_sealion_rerank.ipynb`'s
benchmark (MRR 0.950), beating both `ms-marco-MiniLM-L-6-v2` (MRR 0.425, worse
than no reranking at all) and the intermediate `bge-reranker-v2-m3` baseline
(MRR 0.817) at effectively the same latency cost as the latter. Unlike those
two, SEA-LION is a bi-encoder reranking by cosine similarity, not a
cross-encoder — see Step 2 for the different scoring mechanism this requires.

## Step 1: Setup

Imports, reconnect to the Chroma collection, load `chunks.json` /
`embeddings_bge_m3.npy` / `chunk_ids_bge_m3.json`, and constants (Ollama model
name, base URL).

In [ ]:
import json
from pathlib import Path

import chromadb
import numpy as np

CHUNKS_PATH = Path("../data/processed/chunks.json")
EMBEDDINGS_PATH = Path("../data/processed/embeddings_bge_m3.npy")
CHUNK_IDS_PATH = Path("../data/processed/chunk_ids_bge_m3.json")
CHROMA_DIR = Path("../data/processed/chroma")

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
# Final reranker choice -- 04b_retrieval_sealion_rerank.ipynb's benchmark showed
# SEA-LION (MRR 0.950) beats both ms-marco-MiniLM-L-6-v2 (MRR 0.425) and the
# intermediate bge-reranker-v2-m3 baseline (MRR 0.817), at effectively the same
# latency cost as the latter. It's a bi-encoder (cosine similarity), not a
# cross-encoder like the other two -- see Step 2 for how reranking differs as
# a result.
RERANKER_MODEL_NAME = "aisingapore/SEA-LION-E5-Embedding-600M"
OLLAMA_MODEL_NAME = "qwen3:8b"
OLLAMA_BASE_URL = "http://localhost:11434"

chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
chunks_by_id = {chunk["chunk_id"]: chunk for chunk in chunks}

embeddings = np.load(EMBEDDINGS_PATH)
chunk_ids = json.loads(CHUNK_IDS_PATH.read_text(encoding="utf-8"))
chunk_id_to_idx = {cid: i for i, cid in enumerate(chunk_ids)}

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection(name="bge_m3")

len(chunks), embeddings.shape, collection.count()

## Step 2: Retrieve context

A swappable `retrieve_context(query, method="dense")` wrapping the retrieval
functions built in `04_retrieval` (dense / hybrid / hybrid+rerank), returning the
top-k chunk texts to use as context.

In [ ]:
import re

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
reranker_model = SentenceTransformer(RERANKER_MODEL_NAME)


def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())


bm25 = BM25Okapi([tokenize(chunks_by_id[cid]["text"]) for cid in chunk_ids])


def _dense_retrieve(query: str, top_k: int) -> list[str]:
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)
    results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=top_k)
    return list(results["ids"][0])


def _bm25_retrieve(query: str, top_k: int) -> list[str]:
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(zip(chunk_ids, scores), key=lambda x: x[1], reverse=True)[:top_k]
    return [cid for cid, _ in ranked]


def _hybrid_retrieve(query: str, top_k: int) -> list[str]:
    dense_ids = _dense_retrieve(query, top_k=10)
    bm25_ids = _bm25_retrieve(query, top_k=10)
    scores: dict[str, float] = {}
    for ranked in (dense_ids, bm25_ids):
        for rank, cid in enumerate(ranked, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (60 + rank)
    return [cid for cid, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]]


def _hybrid_rerank_retrieve(query: str, top_k: int) -> list[str]:
    candidates = _hybrid_retrieve(query, top_k=10)
    # STS is the only named prompt documented on the SEA-LION-E5 model card,
    # used for both query and passage since this only needs a symmetric
    # cosine-similarity score for reranking, not asymmetric retrieval.
    query_embedding = reranker_model.encode(query, convert_to_numpy=True, prompt_name="STS")
    candidate_embeddings = reranker_model.encode(
        [chunks_by_id[cid]["text"] for cid in candidates],
        convert_to_numpy=True,
        prompt_name="STS",
    )
    similarities = candidate_embeddings @ query_embedding / (
        np.linalg.norm(candidate_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )
    reranked = sorted(zip(candidates, similarities), key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in reranked[:top_k]]


RETRIEVAL_METHODS = {
    "dense": _dense_retrieve,
    "hybrid": _hybrid_retrieve,
    "hybrid_rerank": _hybrid_rerank_retrieve,
}


def retrieve_context(query: str, method: str = "hybrid_rerank", top_k: int = 5) -> list[dict]:
    retrieved_ids = RETRIEVAL_METHODS[method](query, top_k)
    return [chunks_by_id[cid] for cid in retrieved_ids]


retrieve_context("How much overtime pay am I entitled to?")

## Step 3: Prompt construction

System prompt + retrieved chunks as context + the user's query. Explicit
instruction to answer only from the provided context and say so when the context
doesn't cover the question — hallucinating a wrong salary threshold or deadline is
a real harm here, not just an inconvenience.

In [3]:
SYSTEM_PROMPT = """You are migrantBuddy, an assistant that answers questions about \
Singapore employment rules (work passes, salary, working hours) for migrant workers.

Answer ONLY using the information in the provided context. If the context does not \
contain enough information to answer the question, say so clearly instead of \
guessing. Do not use any outside knowledge. Keep answers clear and concise, \
suitable for someone who may not be a native English speaker."""


def build_prompt(query: str, context_chunks: list[dict]) -> str:
    context_text = "\n\n---\n\n".join(
        f"Source: {chunk['url']}\n{chunk['text']}" for chunk in context_chunks
    )
    return f"""Context:
{context_text}

Question: {query}

Answer:"""


context_chunks = retrieve_context("How much overtime pay am I entitled to?")
prompt = build_prompt("How much overtime pay am I entitled to?", context_chunks)
print(prompt[:1000])

Context:
Source: https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days
Hours of work, overtime and rest day > Overtime pay

Overtime work is all work in excess of the normal hours of work (excluding breaks).

You can claim overtime if you are:

- A non-workman earning a monthly basic salary of $2,600 or less.
- A workman earning a monthly basic salary of $4,500 or less. 

The overtime rate payable for non-workmen is **capped at the salary level of $2,600, or an hourly rate of $13.60**.

For overtime work, your employer must pay you **at least 1.5 times** the hourly basic rate of pay. Payment must be made **within 14 days** after the last day of the salary period.

A non-workman earns $2,600 a month and works 2 hours of overtime. The overtime pay is:

$13.60 × 1.5 × 2 hours = $40.80

Calculate your overtime pay

Overtime pay is calculated as follows:

- Hourly basic rate of pay × 1.5 × number of hours worked overtime

The hourly basic rate of pay is calculated

## Step 4: Ollama call

Send the constructed prompt to the local Ollama server and get the generated
answer back.

In [4]:
import requests


def generate(query: str, method: str = "hybrid_rerank", top_k: int = 5) -> dict:
    context_chunks = retrieve_context(query, method=method, top_k=top_k)
    user_prompt = build_prompt(query, context_chunks)

    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL_NAME,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            "stream": False,
        },
        timeout=120,
    )
    response.raise_for_status()
    answer = response.json()["message"]["content"]

    return {
        "query": query,
        "answer": answer,
        "sources": [chunk["url"] for chunk in context_chunks],
    }


result = generate("How much overtime pay am I entitled to?")
print(result["answer"])

To calculate your overtime pay, follow these steps:  

1. **Eligibility**:  
   - **Non-workmen**: Monthly salary ≤ $2,600 (overtime capped at $13.60/hour).  
   - **Workmen**: Monthly salary ≤ $4,500 (overtime rate = 1.5 × hourly basic rate).  

2. **Hourly Basic Rate Calculation**:  
   - **Monthly-rated**: (12 × monthly salary) ÷ (52 × 44).  
   - **Daily-rated**: Daily pay ÷ working hours per day.  
   - **Piece-rated**: Weekly pay ÷ total hours worked in the week.  

3. **Overtime Rate**:  
   - Minimum pay = **1.5 × hourly basic rate × overtime hours**.  
   - For non-workmen, **capped at $13.60/hour** (e.g., 2 hours = $13.60 × 1.5 × 2 = **$40.80**).  

4. **Limits**:  
   - Maximum overtime per month: **72 hours** (exemption required for more).  
   - Overtime on rest days/public holidays includes **1.5× hourly rate + rest day pay**.  

For exact amounts, calculate based on your salary type and hours worked. Always ensure overtime is paid within **14 days** of the salary period.

## Step 5: End-to-end smoke test

Run the sample queries through retrieve → prompt → generate, and read the answers
for groundedness (does it match the retrieved context) and honesty (does it admit
when context is thin, rather than filling the gap with a guess).

In [5]:
# Same rough, unverified sample queries as 04_retrieval -- smoke-testing only.
SAMPLE_QUERIES = [
    {"language": "en", "text": "How much overtime pay am I entitled to?"},
    {"language": "en", "text": "When must my employer pay my salary?"},
    {"language": "ms", "text": "Bilakah majikan saya perlu bayar gaji saya?"},
    {"language": "ta", "text": "எனக்கு எவ்வளவு கூடுதல் நேர ஊதியம் கிடைக்கும்?"},
    {"language": "my", "text": "ကျွန်တော် ဘယ်လောက် အချိန်ပိုခ ရထိုက်သလဲ"},
    {"language": "th", "text": "ฉันมีสิทธิ์ได้รับค่าล่วงเวลาเท่าไหร่"},
    {"language": "vi", "text": "Chủ sử dụng lao động của tôi phải trả lương khi nào?"},
    {"language": "en", "text": "Who pays repatriation costs when my Work Permit ends?"},
    {"language": "en", "text": "How much medical insurance must my employer provide?"},
    {"language": "en", "text": "How can I contact MOM?"},
]

# Full answers kept here (nothing lost) -- print output below is truncated per
# query, since printing 10 full answers in one cell overflows VSCode's output
# display. Inspect smoke_test_results[i]["answer"] directly for the full text.
smoke_test_results = []
for query in SAMPLE_QUERIES:
    result = generate(query["text"])
    result["language"] = query["language"]
    smoke_test_results.append(result)

    preview = result["answer"][:300].replace("\n", " ")
    print(f"\n{'=' * 80}\nQuery [{query['language']}]: {query['text']}\n{'=' * 80}")
    print(f"\nAnswer (first 300 chars):\n{preview}...")
    print(f"\nSources: {result['sources']}")


Query [en]: How much overtime pay am I entitled to?

Answer (first 300 chars):
To calculate your overtime pay, follow these steps:    1. **Check your salary level**:      - **Non-workman**: Monthly basic salary ≤ $2,600.      - **Workman**: Monthly basic salary ≤ $4,500.    2. **Calculate your hourly basic rate**:      - **Monthly-rated employee**:        $ \text{Hourly rate} ...

Sources: ['https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days', 'https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days', 'https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days', 'https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days', 'https://www.mom.gov.sg/employment-practices/salary/paying-salary']

Query [en]: When must my employer pay my salary?

Answer (first 300 chars):
Your employer must pay your salary **at least once a month**, **within 7 days after the end of the salary period**. For **ov